In [1]:
from segment_anything import SamPredictor, sam_model_registry
import os
import cv2
import torch
import requests
import numpy as np
from PIL import Image
import os
import torch
import random
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

In [2]:
csv_dir = "/users/PAS2136/rayees/ML-Challenge/Detection-GDINO/Test-for-SAM"

test_images = [f for f in os.listdir(csv_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff'))]


csv_path = f"{csv_dir}/SAM_test_input.csv"

df = pd.read_csv(csv_path)
df.head(2)


,filename,actual,detected,bounding_boxes
0,Carabus_serratus-Btray-Y2018-NEON.BET.D09.0030...,13,13,"[[2731.078369140625, 2394.053466796875, 3029.4..."
1,Pterostichus_permundus-Btray-Y2021-NEON.BET.D0...,23,23,"[[553.7706909179688, 3187.294921875, 808.65045..."


In [3]:
image_dir = "/fs/ess/PAS2136/CarabidImaging/Images/FinalImages/ABTrays/"

img_paths = []

files = os.listdir(image_dir)


for img in files : 
    impath = os.path.join(image_dir, f"{img}")
    if os.path.exists(impath) and img in test_images : 
        img_paths.append(impath)
        

random.shuffle(img_paths)

In [4]:
url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"

SAM_model_path = "/users/PAS2136/rayees/SAM/sam_vit_h_4b8939.pth"

if os.path.exists(SAM_model_path) == False : 
    response = requests.get(url)
    with open(SAM_model_path, 'wb') as f:
        f.write(response.content)


In [5]:
# Load SAM model

model_type = "vit_h"  # You can choose from other model types: "sam_vit_b", "sam_vit_l"

sam = sam_model_registry[model_type](checkpoint=SAM_model_path)

sam.to(device="cuda" if torch.cuda.is_available() else "cpu")

predictor = SamPredictor(sam)


In [6]:
import ast

def parse_bboxes(bboxes):
    if isinstance(bboxes, str):
        try:
            bboxes = ast.literal_eval(bboxes)  # Convert string to list
        except (ValueError, SyntaxError) as e:
            print(f"Error parsing bounding boxes: {e}")
            return []
    if not isinstance(bboxes, list):
        print(f"Invalid bounding boxes format: {bboxes}")
        return []
    # Ensure each bbox is a list of 4 floats
    valid_bboxes = []
    for bbox in bboxes:
        if isinstance(bbox, list) and len(bbox) == 4 and all(isinstance(x, (int, float)) for x in bbox):
            valid_bboxes.append(bbox)
        else:
            print(f"Skipping invalid bbox: {bbox}")
    return valid_bboxes

In [ ]:
# Process each image
for img_path in img_paths:
    # Extract image filename without extension
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    
    # Create output directory for this image
    output_dir = f"{img_name}_segments"
    os.makedirs(output_dir, exist_ok=True)
    
    # Load image
    try:
        image = Image.open(img_path).convert("RGB")
        image_np = np.array(image)
    except Exception as e:
        print(f"Error loading image {img_name}: {str(e)}")
        continue
    
    # Set image in predictor
    try:
        predictor.set_image(image_np)
    except Exception as e:
        print(f"Error setting image {img_name} in predictor: {str(e)}")
        continue
    
    # Find corresponding row in dataframe
    df_row = df[df['filename'].str.contains(img_name)]
    
    if df_row.empty:
        print(f"No bounding boxes found for {img_name}, skipping...")
        continue
    
    # Get and parse bounding boxes
    bboxes = parse_bboxes(df_row['bounding_boxes'].iloc[0])
    
    if not bboxes:
        print(f"No valid bounding boxes for {img_name}, skipping...")
        continue
    
    # Process each bounding box
    for idx, bbox in enumerate(bboxes):
        # bbox format: [x_min, y_min, x_max, y_max]
        try:
            bbox_np = np.array([bbox[0], bbox[1], bbox[2], bbox[3]])
            print(f"Processing bbox {idx}: {bbox_np}")
            
            # Run SAM with bounding box prompt
            masks, scores, logits = predictor.predict(
                box=bbox_np,
                multimask_output=False
            )
            
            # Save segmentation mask
            mask = masks[0]  # Take the first mask
            mask_img = Image.fromarray((mask * 255).astype(np.uint8))
            mask_path = os.path.join(output_dir, f"mask_{idx:03d}.png")
            mask_img.save(mask_path)
            
            # Optional: Save visualization
            plt.figure(figsize=(10, 10))
            plt.imshow(image_np)
            plt.imshow(mask, alpha=0.5, cmap='jet')
            plt.axis('off')
            vis_path = os.path.join(output_dir, f"vis_mask_{idx:03d}.png")
            plt.savefig(vis_path, bbox_inches='tight')
            plt.close()
            
        except Exception as e:
            print(f"Error processing bbox {idx} for {img_name}: {str(e)}")
            continue
    
    print(f"Processed {len(bboxes)} masks for {img_name}, saved to {output_dir}")

# Optional: Save metadata
metadata = {
    "images_processed": len(img_paths),
    "output_directories": [f"{os.path.splitext(os.path.basename(img_path))[0]}_segments" for img_path in img_paths]
}
with open("processing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("All images processed successfully!")

Processing bbox 0: [2731.07836914 2394.0534668  3029.43408203 2895.87792969]
Processing bbox 1: [1975.40148926 2402.89916992 2239.01220703 2941.33349609]
Processing bbox 2: [1541.87646484 2367.42358398 1885.5559082  2923.87695312]
Processing bbox 3: [1963.24658203 3015.79321289 2313.58984375 3633.99194336]
Processing bbox 4: [3053.20776367 2379.31787109 3434.27929688 2913.99194336]
Processing bbox 5: [ 531.32958984 2379.45361328  803.17266846 2848.00561523]
Processing bbox 6: [1636.91394043 3014.30297852 1887.24829102 3520.33422852]
Processing bbox 7: [1208.5802002  3053.9675293  1513.39941406 3601.55957031]
Processing bbox 8: [1179.83569336 2387.55249023 1501.5970459  2922.56030273]
Processing bbox 9: [2296.6171875  2275.34619141 2660.67407227 3017.38964844]
Processing bbox 10: [ 891.8894043  2310.9855957  1148.49853516 2978.47705078]
Processing bbox 11: [ 554.15734863 3020.17236328  865.24072266 3618.89526367]
Processing bbox 12: [ 848.37994385 3068.77709961 1134.50561523 3576.291503